# 02 — Residue Structural Features

In this step, I annotate each residue in hen egg-white lysozyme (1LYZ) with structure-aware features derived from DSSP.

These features include:
- secondary structure
- solvent accessibility
- relative solvent exposure
- residue environment class (core, intermediate, surface)

This information will later be used to choose mutation sites and interpret likely stability effects.

In [23]:
from pathlib import Path
import os
import pandas as pd
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP

***Set DSSP dictionary path***

In [24]:
os.environ["LIBCIFPP_DATA_DIR"] = "/Users/vaibhav.mh/miniconda3/envs/mutation-effect-prediction/var/cache/libcifpp"

print("LIBCIFPP_DATA_DIR set to:")
print(os.environ["LIBCIFPP_DATA_DIR"])

LIBCIFPP_DATA_DIR set to:
/Users/vaibhav.mh/miniconda3/envs/mutation-effect-prediction/var/cache/libcifpp


***load the structure again***

In [25]:
pdb_path = Path("../data/input/1LYZ.pdb").resolve()

parser = PDBParser(QUIET=True)
structure = parser.get_structure("1LYZ", str(pdb_path))

model = structure[0]
chain = model["A"]

print("Structure loaded successfully")
print("PDB path:", pdb_path)
print("Chain:", chain.id)

Structure loaded successfully
PDB path: /Users/vaibhav.mh/protein_repos/mutation-effect-prediction/data/input/1LYZ.pdb
Chain: A


***Run DSSP***

In [26]:
mkdssp_path = "/Users/vaibhav.mh/miniconda3/envs/mutation-effect-prediction/bin/mkdssp"

dssp = DSSP(model, str(pdb_path), dssp=mkdssp_path)

print("DSSP worked")
print("Number of DSSP entries:", len(dssp))

DSSP worked
Number of DSSP entries: 129


/Users/vaibhav.mh/miniconda3/envs/mutation-effect-prediction/lib/python3.10/site-packages/Bio/PDB/DSSP.py:199: UserWarning: parse error at line 1: This file does not seem to be an mmCIF file

  warnings.warn(err)


dssp_keys = list(dssp.keys())

print("First 5 DSSP keys and entries:\n")
for key in dssp_keys[:5]:
    print("KEY:", key)
    print("VALUE:", dssp[key])
    print()

***Inspect a few DSSP entries***

In [27]:
dssp_keys = list(dssp.keys())

print("First 5 DSSP keys and entries:\n")
for key in dssp_keys[:5]:
    print("KEY:", key)
    print("VALUE:", dssp[key])
    print()

First 5 DSSP keys and entries:

KEY: ('A', (' ', 1, ' '))
VALUE: (1, 'K', '-', 0.47317073170731705, 360.0, 122.5, 0, 0.0, 39, -2.4, 0, 0.0, 2, -0.5)

KEY: ('A', (' ', 2, ' '))
VALUE: (2, 'V', 'B', 0.6267605633802817, -98.7, 126.9, 37, -0.2, 37, -0.3, 38, -0.1, 84, -0.0)

KEY: ('A', (' ', 3, ' '))
VALUE: (3, 'F', '-', 0.09644670050761421, -73.0, 166.2, 35, -3.5, 2, -0.2, -2, -0.5, 3, -0.0)

KEY: ('A', (' ', 4, ' '))
VALUE: (4, 'G', '-', 0.34523809523809523, -111.1, 159.0, 1, -0.2, 4, -2.4, -2, -0.0, 5, -0.1)

KEY: ('A', (' ', 5, ' '))
VALUE: (5, 'R', 'H', 0.31451612903225806, -56.4, -77.0, 2, -0.2, 4, -1.5, 3, -0.2, -1, -0.2)



***Extract residue features from DSSP***

In [29]:
feature_rows = []

for key in dssp.keys():
    chain_id, residue_id = key
    hetflag, resseq, icode = residue_id

    if chain_id != "A":
        continue

    entry = dssp[key]

    aa = entry[1]
    ss = entry[2]
    rsa = entry[3]   # use this directly; do NOT divide again

    feature_rows.append({
        "chain": chain_id,
        "position": resseq,
        "insertion_code": icode.strip() if isinstance(icode, str) else "",
        "aa_dssp": aa,
        "secondary_structure": ss,
        "rsa": rsa
    })

features_df = pd.DataFrame(feature_rows)
features_df.head()

,chain,position,insertion_code,aa_dssp,secondary_structure,rsa
0,A,1,,K,-,0.473171
1,A,2,,V,B,0.626761
2,A,3,,F,-,0.096447
3,A,4,,G,-,0.345238
4,A,5,,R,H,0.314516


***Simplify the secondary structure***

In [31]:
def simplify_ss(ss_code):
    if ss_code == "H":
        return "helix"
    elif ss_code == "E":
        return "sheet"
    else:
        return "loop"

features_df["ss_simple"] = features_df["secondary_structure"].apply(simplify_ss)
features_df.head()

,chain,position,insertion_code,aa_dssp,secondary_structure,rsa,ss_simple
0,A,1,,K,-,0.473171,loop
1,A,2,,V,B,0.626761,loop
2,A,3,,F,-,0.096447,loop
3,A,4,,G,-,0.345238,loop
4,A,5,,R,H,0.314516,helix


***Classify environment***

In [33]:
def classify_environment(rsa):
    if pd.isna(rsa):
        return "unknown"
    elif rsa < 0.10:
        return "core"
    elif rsa <= 0.25:
        return "intermediate"
    else:
        return "surface"

features_df["environment"] = features_df["rsa"].apply(classify_environment)
features_df.head(50)

,chain,position,insertion_code,aa_dssp,secondary_structure,rsa,ss_simple,environment
0,A,1,,K,-,0.473171,loop,surface
1,A,2,,V,B,0.626761,loop,surface
2,A,3,,F,-,0.096447,loop,core
3,A,4,,G,-,0.345238,loop,surface
4,A,5,,R,H,0.314516,helix,surface
5,A,6,,C,H,0.370370,helix,surface
6,A,7,,E,H,0.463918,helix,surface
7,A,8,,L,H,0.000000,helix,core
8,A,9,,A,H,0.000000,helix,core
9,A,10,,A,H,0.443396,helix,surface


***Load the cleaned residue table***

In [34]:
residue_df = pd.read_csv("../data/processed/residue_table.csv")
residue_df.head()

,chain,position,insertion_code,resname_3,resname_1
0,A,1,NaN,LYS,K
1,A,2,NaN,VAL,V
2,A,3,NaN,PHE,F
3,A,4,NaN,GLY,G
4,A,5,NaN,ARG,R


***Merge the tables***

In [35]:
merged_df = residue_df.merge(
    features_df[["position", "aa_dssp", "secondary_structure", "ss_simple", "rsa", "environment"]],
    on="position",
    how="left"
)

merged_df.head()

,chain,position,insertion_code,resname_3,resname_1,aa_dssp,secondary_structure,ss_simple,rsa,environment
0,A,1,NaN,LYS,K,K,-,loop,0.473171,surface
1,A,2,NaN,VAL,V,V,B,loop,0.626761,surface
2,A,3,NaN,PHE,F,F,-,loop,0.096447,core
3,A,4,NaN,GLY,G,G,-,loop,0.345238,surface
4,A,5,NaN,ARG,R,R,H,helix,0.314516,surface


***Basic quality checks / check counts***

In [36]:
print("Total residues in residue table:", len(residue_df))
print("Total residues after merge:", len(merged_df))

print("\nEnvironment counts:")
print(merged_df["environment"].value_counts(dropna=False))

print("\nSecondary structure counts:")
print(merged_df["ss_simple"].value_counts(dropna=False))

Total residues in residue table: 129
Total residues after merge: 129

Environment counts:
environment
surface         65
core            38
intermediate    26
Name: count, dtype: int64

Secondary structure counts:
ss_simple
loop     84
helix    37
sheet     8
Name: count, dtype: int64


***Save the feature tables***

In [37]:
output_path = Path("../data/processed/residue_features.csv")
merged_df.to_csv(output_path, index=False)

print(f"Saved cleaned residue features to: {output_path}")

Saved cleaned residue features to: ../data/processed/residue_features.csv


In [38]:
print(merged_df["environment"].value_counts(dropna=False))

environment
surface         65
core            38
intermediate    26
Name: count, dtype: int64


In [39]:
merged_df.head()

,chain,position,insertion_code,resname_3,resname_1,aa_dssp,secondary_structure,ss_simple,rsa,environment
0,A,1,NaN,LYS,K,K,-,loop,0.473171,surface
1,A,2,NaN,VAL,V,V,B,loop,0.626761,surface
2,A,3,NaN,PHE,F,F,-,loop,0.096447,core
3,A,4,NaN,GLY,G,G,-,loop,0.345238,surface
4,A,5,NaN,ARG,R,R,H,helix,0.314516,surface
